## 1. Import Libraries
Import all required Python libraries for data analysis, geospatial calculations, and visualization.

In [66]:
#import statements
import pandas as pd
import numpy as np
import altair as alt
import geopandas as gpd
import colorsys
from ipyleaflet import Map, CircleMarker, Polygon, basemaps, WidgetControl
from ipywidgets import HTML
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
from geopy.distance import geodesic

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsRegressor

## 2. Data Loading Functions
Define functions to load food security, port, and centroid data.

In [67]:
# Function to load and do some minor preprocessing on data
def load_data():
    # 1. Food Security Data
    food_df = pd.read_csv("data/undernourishment_raw.csv")
    # Process the raw World Bank export
    year_cols = [c for c in food_df.columns if c.startswith('YR')]
    food_df['latest_undernourishment'] = food_df[year_cols].bfill(axis=1).iloc[:, 0]
    food_df = food_df[['Country', 'latest_undernourishment']]
    print(food_df.head())
    # 2. Major Ports Data
    ports_df = pd.read_csv("data/major_ports.csv")
    # 3. Country Centroids Data
    centroids_df = pd.read_csv("data/country_centroids.csv")
    return food_df, ports_df, centroids_df

## 3. Distance Calculation Function
Define a function to calculate the distance from each country to its nearest major port.

In [68]:
# Function to calculate distance to nearest port for each country
def calculate_nearest_port_distance(country_df, ports_df):
    def get_min_dist(row):
        # Coordinates in country_centroids.csv are 'latitude' and 'longitude'
        country_coord = (row['latitude'], row['longitude'])
        # Coordinates in major_ports.csv are 'lat' and 'lon'
        distances = [geodesic(country_coord, (p['lat'], p['lon'])).km for _, p in ports_df.iterrows()]
        return min(distances)
    country_df['dist_to_nearest_port_km'] = country_df.apply(get_min_dist, axis=1)
    return country_df

## 4. Load Data
Load the food security, ports, and centroid data using the defined function.

In [69]:
# Load data
try:
    food_df, ports_df, centroids_df = load_data()
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure the 'data' folder contains the required CSV files.")

                 Country  latest_undernourishment
0               Zimbabwe                     30.0
1                 Zambia                     32.1
2            Yemen, Rep.                      NaN
3     West Bank and Gaza                      NaN
4  Virgin Islands (U.S.)                      NaN


## 5. Merge DataFrames
Merge the food security data with country centroids for further analysis.

In [70]:
# Note: Using 'left' merge to keep all food data, then dropping countries without coordinates
merged_df = pd.merge(food_df, centroids_df, left_on='Country', right_on='name')

## 6. Calculate Distance to Nearest Port
Add a column for the distance from each country to its nearest major port.

In [71]:
final_df = calculate_nearest_port_distance(merged_df, ports_df)

## 7. Visualize Results
Create a scatter plot showing the relationship between distance to port and undernourishment.

In [72]:
# Visualization: Scatter plot of distance to nearest port vs. undernourishment.
# Our hypothesis is that countries farther from major ports may have higher undernourishment due to supply chain challenges.

chart = alt.Chart(final_df).mark_point(filled=True, size=60).encode(
    x=alt.X('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('dist_to_nearest_port_km:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'dist_to_nearest_port_km', 'latest_undernourishment']
).properties(
    width=600,
    height=400,
    title='Impact of Supply Chain Proximity on Food Security'
).interactive()
chart


alt.Chart(...)

In [73]:
# Pearson Correlation for Distance to Nearest Port vs Undernourishment
correlation = final_df['dist_to_nearest_port_km'].corr(final_df['latest_undernourishment'])
print(f"Pearson correlation: {correlation:.3f}")

Pearson correlation: 0.113


In [74]:
# Map developed with assistance from GitHub Copilot

# Interactive map showing shaded countries and major ports as dots
# Load world country boundaries from Natural Earth via URL
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# Check available columns and standardize
if 'NAME' in world.columns:
    world['name'] = world['NAME']
elif 'ADMIN' in world.columns:
    world['name'] = world['ADMIN']

# Merge world geometries with our analysis data by country name
world_data = world.merge(final_df[['Country', 'dist_to_nearest_port_km', 'latest_undernourishment', 'latitude', 'longitude']],
                         left_on='name', right_on='Country', how='left')

# Create map
map_center = (final_df['latitude'].mean(), final_df['longitude'].mean())
m = Map(center=map_center, zoom=2, basemap=basemaps.CartoDB.Positron)

# Create colormaps for distance and undernourishment visualization
dist_min = final_df['dist_to_nearest_port_km'].min()
dist_max = final_df['dist_to_nearest_port_km'].max()
und_min = final_df['latest_undernourishment'].min()
und_max = final_df['latest_undernourishment'].max()
dist_norm = Normalize(vmin=dist_min, vmax=dist_max)
und_norm = Normalize(vmin=und_min, vmax=und_max)

# Use hue for undernourishment and brightness/value for distance.
def get_bivariate_color(distance, undernourishment):
    if pd.isna(distance) or pd.isna(undernourishment):
        return '#cccccc'
    # Under-nourishment -> hue: lower = blue/green, higher = red
    hue = 0.55 - 0.4 * und_norm(undernourishment)
    hue = np.clip(hue, 0.0, 1.0)
    # Distance -> value: closer countries are brighter, farther are darker
    value = 0.4 + 0.6 * (1 - dist_norm(distance))
    value = np.clip(value, 0.0, 1.0)
    saturation = 0.9
    r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
    return '#{:02x}{:02x}{:02x}'.format(
        int(r * 255),
        int(g * 255),
        int(b * 255)
    )

# Add country polygons shaded by both metrics
for idx, row in world_data.iterrows():
    if pd.notna(row['dist_to_nearest_port_km']) and pd.notna(row['latest_undernourishment']):
        color_hex = get_bivariate_color(
            row['dist_to_nearest_port_km'],
            row['latest_undernourishment']
        )

        # Create popup with country info
        popup_text = (
            f"<b>Country:</b> {row['Country']}<br>"
            f"<b>Distance to nearest port:</b> {row['dist_to_nearest_port_km']:.1f} km<br>"
            f"<b>Undernourishment:</b> {row['latest_undernourishment']} %"
        )

        # Handle both Polygon and MultiPolygon geometries
        if row['geometry'].geom_type == 'Polygon':
            # Convert (lon, lat) from geopandas to (lat, lon) for ipyleaflet
            coords = [(lat, lon) for lon, lat in row['geometry'].exterior.coords]
            poly = Polygon(
                locations=coords,
                color='gray',
                weight=1,
                fill_color=color_hex,
                fill_opacity=0.7,
                popup=HTML(popup_text)
            )
            m.add_layer(poly)
        elif row['geometry'].geom_type == 'MultiPolygon':
            # Add each polygon in the multipolygon
            for geom in row['geometry'].geoms:
                # Convert (lon, lat) from geopandas to (lat, lon) for ipyleaflet
                coords = [(lat, lon) for lon, lat in geom.exterior.coords]
                poly = Polygon(
                    locations=coords,
                    color='gray',
                    weight=1,
                    fill_color=color_hex,
                    fill_opacity=0.7,
                    popup=HTML(popup_text)
                )
                m.add_layer(poly)

# Add major ports as simple red dots
for _, port in ports_df.iterrows():
    port_marker = CircleMarker(
        location=(port['lat'], port['lon']),
        radius=4,
        color='red',
        weight=2,
        fill=True,
        fill_color='red',
        fill_opacity=0.8,
        popup=HTML(f"<b>{port.get('port_name', 'Major Port')}</b>")
    )
    m.add_layer(port_marker)

# Add a legend describing the bivariate shading and port markers
legend_html = HTML(
    "<div style='background: rgba(255,255,255,0.9); color: black; padding: 10px; border-radius: 8px; font-size: 12px; line-height: 1.5;'>"
    "<b>Map legend</b><br>"
    "<span style='display:block; margin-top:6px;'><span style='background:#ff7f7f; width:12px; height:12px; display:inline-block; margin-right:6px;'></span>Port marker</span>"
    "<hr style='margin:6px 0;'>"
    "<b>Country fill</b><br>"
    "<span style='display:block; margin-top:6px;'>Hue: undernourishment</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Blue = lower undernourishment</li>"
    "<li>Red = higher undernourishment</li>"
    "</ul>"
    "<span style='display:block; margin-top:6px;'>Brightness: distance to nearest port</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Brighter = closer to a major port</li>"
    "<li>Darker = farther from a major port</li>"
    "</ul>"
    "</div>"
)
legend_control = WidgetControl(widget=legend_html, position='bottomright')
m.add_control(legend_control)

m

Map(center=[np.float64(18.898025440677966), np.float64(14.51414856497175)], controls=(ZoomControl(options=['po…

## 8. Show and Save Processed Data
Display a sample of the processed data and save the results to a CSV file.

In [75]:
#
print("\nSample of Processed Data:")
display(final_df[['Country', 'dist_to_nearest_port_km', 'latest_undernourishment']].head())

# Save the final merged dataset for further analysis
output_file = 'processed_food_security_analysis.csv'
final_df.to_csv(output_file, index=False)
print(f"\nAnalysis ready! Results saved to '{output_file}'")


Sample of Processed Data:


,Country,dist_to_nearest_port_km,latest_undernourishment
0,Zimbabwe,603.957118,30.0
1,Zambia,1052.148811,32.1
2,Vanuatu,1966.503191,7.9
3,Uzbekistan,1845.476744,2.5
4,Uruguay,266.371814,2.5



Analysis ready! Results saved to 'processed_food_security_analysis.csv'


## 9. Import WEO Dataset

In [76]:
# Additional exploratory analysis: Load WEO data to see if we can find any macroeconomic correlations
weo_df = pd.read_csv("data/WEO_Data.xls", sep='\t')
print(weo_df.head())

  WEO Country Code WEO Subject Code              Country  \
0              512            NGDPD          Afghanistan   
1              914            NGDPD              Albania   
2              612            NGDPD              Algeria   
3              614            NGDPD               Angola   
4              311            NGDPD  Antigua and Barbuda   

                       Subject Descriptor         Units     Scale  \
0  Gross domestic product, current prices  U.S. dollars  Billions   
1  Gross domestic product, current prices  U.S. dollars  Billions   
2  Gross domestic product, current prices  U.S. dollars  Billions   
3  Gross domestic product, current prices  U.S. dollars  Billions   
4  Gross domestic product, current prices  U.S. dollars  Billions   

                       Country/Series-specific Notes     2016     2017  \
0  See notes for:  Gross domestic product, curren...   18.020   18.883   
1  See notes for:  Gross domestic product, curren...   11.862   13.053   
2 

## 10. Filter for Developing Countries in WEO

In [77]:
developing_countries = weo_df['Country'].unique()
filtered_df = final_df[final_df['Country'].isin(developing_countries)]
print(f"Number of developing countries in analysis: {len(filtered_df)}")

Number of developing countries in analysis: 127


## 11. Repeat Comparison Analysis on Developing Countries

In [78]:
# Recreate the scatter plot for just developing countries to see if the relationship is stronger
chart_developing = alt.Chart(filtered_df).mark_point(filled=True, size=60).encode(
    x=alt.X('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('dist_to_nearest_port_km:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'dist_to_nearest_port_km', 'latest_undernourishment']
).properties(
    width=600,
    height=400,
    title='Impact of Supply Chain Proximity on Food Security (Developing Countries)'
).interactive()
chart_developing


alt.Chart(...)

In [79]:
# Pearson Correlation for Distance to Nearest Port vs Undernourishment, for Developing Countries
correlation = filtered_df['dist_to_nearest_port_km'].corr(filtered_df['latest_undernourishment'])
print(f"Pearson correlation: {correlation:.3f}")

Pearson correlation: 0.035


## 12. Extract GDP from WEO

In [80]:
#Also check GDP to see if we can find any macroeconomic correlations
gdp_df = weo_df[(weo_df['Subject Descriptor'] == 'Gross domestic product, current prices') &
                (weo_df['Units'] == 'U.S. dollars') &
                (weo_df['Scale'] == 'Billions')][['Country', '2024']]
gdp_df.rename(columns={'2024': 'GDP_billions_USD'}, inplace=True)
print(gdp_df.head())

               Country GDP_billions_USD
0          Afghanistan              NaN
1              Albania           20.847
2              Algeria          210.860
3               Angola          124.102
4  Antigua and Barbuda            2.016


## 13. Merge GDP with Analysis Data

In [81]:
merged_gdp_df = pd.merge(filtered_df, gdp_df, on='Country', how='inner')
merged_gdp_df['GDP_billions_USD'] = pd.to_numeric(merged_gdp_df['GDP_billions_USD'], errors='coerce')
print(f"Number of countries with GDP data: {len(merged_gdp_df)}")

Number of countries with GDP data: 127


## 14. GDP vs Undernourishment

In [82]:
# Scatter plot of GDP vs undernourishment for developing countries
chart_gdp_und = alt.Chart(merged_gdp_df).mark_point(filled=True, size=60).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('GDP_billions_USD:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'GDP_billions_USD', 'latest_undernourishment']
).properties(
    width=600,
    height=400,
    title='GDP vs Undernourishment in Developing Countries'
).interactive()
chart_gdp_und

alt.Chart(...)

In [83]:
# Pearson Correlation for GDP vs Undernourishment, for Developing Countries
correlation = merged_gdp_df['GDP_billions_USD'].corr(merged_gdp_df['latest_undernourishment'])
print(f"Pearson correlation: {correlation:.3f}")

Pearson correlation: -0.261


In [84]:
# Map developed with assistance from GitHub Copilot

# Merge world geometries with GDP and undernourishment data
world_gdp_und = world.merge(merged_gdp_df[['Country', 'GDP_billions_USD', 'latest_undernourishment', 'latitude', 'longitude']],
                            left_on='name', right_on='Country', how='left')

# Create map
map_center = (merged_gdp_df['latitude'].mean(), merged_gdp_df['longitude'].mean())
m_gdp_und = Map(center=map_center, zoom=2, basemap=basemaps.CartoDB.Positron)

# Create normalizers for GDP and undernourishment
gdp_min = merged_gdp_df['GDP_billions_USD'].min()
gdp_max = merged_gdp_df['GDP_billions_USD'].max()
und_min = merged_gdp_df['latest_undernourishment'].min()
und_max = merged_gdp_df['latest_undernourishment'].max()
gdp_norm = Normalize(vmin=gdp_min, vmax=gdp_max)
und_norm = Normalize(vmin=und_min, vmax=und_max)

# Bivariate color: Hue for GDP, Brightness for Undernourishment
def get_gdp_und_color(gdp, undernourishment):
    if pd.isna(gdp) or pd.isna(undernourishment):
        return '#cccccc'
    # GDP -> hue: lower GDP = blue, higher GDP = red
    hue = 0.55 - 0.4 * gdp_norm(gdp)
    hue = np.clip(hue, 0.0, 1.0)
    # Undernourishment -> value: lower = brighter, higher = darker
    value = 0.4 + 0.6 * (1 - und_norm(undernourishment))
    value = np.clip(value, 0.0, 1.0)
    saturation = 0.9
    r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
    return '#{:02x}{:02x}{:02x}'.format(
        int(r * 255),
        int(g * 255),
        int(b * 255)
    )

# Add country polygons
for idx, row in world_gdp_und.iterrows():
    if pd.notna(row['GDP_billions_USD']) and pd.notna(row['latest_undernourishment']):
        color_hex = get_gdp_und_color(row['GDP_billions_USD'], row['latest_undernourishment'])

        popup_text = (
            f"<b>Country:</b> {row['Country']}<br>"
            f"<b>GDP:</b> {row['GDP_billions_USD']:.1f} billion USD<br>"
            f"<b>Undernourishment:</b> {row['latest_undernourishment']} %"
        )

        if row['geometry'].geom_type == 'Polygon':
            coords = [(lat, lon) for lon, lat in row['geometry'].exterior.coords]
            poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
            m_gdp_und.add_layer(poly)
        elif row['geometry'].geom_type == 'MultiPolygon':
            for geom in row['geometry'].geoms:
                coords = [(lat, lon) for lon, lat in geom.exterior.coords]
                poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
                m_gdp_und.add_layer(poly)

# Add ports
for _, port in ports_df.iterrows():
    port_marker = CircleMarker(location=(port['lat'], port['lon']), radius=4, color='red', weight=2, fill=True, fill_color='red', fill_opacity=0.8, popup=HTML(f"<b>{port.get('port_name', 'Major Port')}</b>"))
    m_gdp_und.add_layer(port_marker)

# Add legend
legend_html = HTML(
    "<div style='background: rgba(255,255,255,0.9); color: black; padding: 10px; border-radius: 8px; font-size: 12px; line-height: 1.5;'>"
    "<b>Map legend</b><br>"
    "<span style='display:block; margin-top:6px;'><span style='background:#ff7f7f; width:12px; height:12px; display:inline-block; margin-right:6px;'></span>Port marker</span>"
    "<hr style='margin:6px 0;'>"
    "<b>Country fill</b><br>"
    "<span style='display:block; margin-top:6px;'>Hue: GDP</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Blue = lower GDP</li>"
    "<li>Red = higher GDP</li>"
    "</ul>"
    "<span style='display:block; margin-top:6px;'>Brightness: undernourishment</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Brighter = lower undernourishment</li>"
    "<li>Darker = higher undernourishment</li>"
    "</ul>"
    "</div>"
)
legend_control = WidgetControl(widget=legend_html, position='bottomright')
m_gdp_und.add_control(legend_control)

m_gdp_und

Map(center=[np.float64(11.858591944881889), np.float64(17.16719923622047)], controls=(ZoomControl(options=['po…

## 15. GDP vs Distance to Nearest Port

In [85]:
# Scatter plot of GDP vs distance to nearest port for developing countries
chart_gdp_dist = alt.Chart(merged_gdp_df).mark_point(filled=True, size=60).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    color=alt.Color('GDP_billions_USD:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'GDP_billions_USD', 'dist_to_nearest_port_km']
).properties(
    width=600,
    height=400,
    title='GDP vs Distance to Nearest Port in Developing Countries'
).interactive()
chart_gdp_dist


alt.Chart(...)

In [86]:
# Pearson Correlation for GDP vs Distance to Nearest Port, for Developing Countries
correlation = merged_gdp_df['GDP_billions_USD'].corr(merged_gdp_df['dist_to_nearest_port_km'])
print(f"Pearson correlation: {correlation:.3f}")

Pearson correlation: -0.324


In [87]:
# Map developed with assistance from GitHub Copilot

# Merge world geometries with GDP and distance data
world_gdp_dist = world.merge(merged_gdp_df[['Country', 'GDP_billions_USD', 'dist_to_nearest_port_km', 'latitude', 'longitude']],
                             left_on='name', right_on='Country', how='left')

# Create map
map_center = (merged_gdp_df['latitude'].mean(), merged_gdp_df['longitude'].mean())
m_gdp_dist = Map(center=map_center, zoom=2, basemap=basemaps.CartoDB.Positron)

# Create normalizers for GDP and distance
gdp_min = merged_gdp_df['GDP_billions_USD'].min()
gdp_max = merged_gdp_df['GDP_billions_USD'].max()
dist_min = merged_gdp_df['dist_to_nearest_port_km'].min()
dist_max = merged_gdp_df['dist_to_nearest_port_km'].max()
gdp_norm = Normalize(vmin=gdp_min, vmax=gdp_max)
dist_norm = Normalize(vmin=dist_min, vmax=dist_max)

# Bivariate color: Hue for GDP, Brightness for Distance
def get_gdp_dist_color(gdp, distance):
    if pd.isna(gdp) or pd.isna(distance):
        return '#cccccc'
    # GDP -> hue: lower GDP = blue, higher GDP = red
    hue = 0.55 - 0.4 * gdp_norm(gdp)
    hue = np.clip(hue, 0.0, 1.0)
    # Distance -> value: closer = brighter, farther = darker
    value = 0.4 + 0.6 * (1 - dist_norm(distance))
    value = np.clip(value, 0.0, 1.0)
    saturation = 0.9
    r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
    return '#{:02x}{:02x}{:02x}'.format(
        int(r * 255),
        int(g * 255),
        int(b * 255)
    )

# Add country polygons
for idx, row in world_gdp_dist.iterrows():
    if pd.notna(row['GDP_billions_USD']) and pd.notna(row['dist_to_nearest_port_km']):
        color_hex = get_gdp_dist_color(row['GDP_billions_USD'], row['dist_to_nearest_port_km'])

        popup_text = (
            f"<b>Country:</b> {row['Country']}<br>"
            f"<b>GDP:</b> {row['GDP_billions_USD']:.1f} billion USD<br>"
            f"<b>Distance to nearest port:</b> {row['dist_to_nearest_port_km']:.1f} km"
        )

        if row['geometry'].geom_type == 'Polygon':
            coords = [(lat, lon) for lon, lat in row['geometry'].exterior.coords]
            poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
            m_gdp_dist.add_layer(poly)
        elif row['geometry'].geom_type == 'MultiPolygon':
            for geom in row['geometry'].geoms:
                coords = [(lat, lon) for lon, lat in geom.exterior.coords]
                poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
                m_gdp_dist.add_layer(poly)

# Add ports
for _, port in ports_df.iterrows():
    port_marker = CircleMarker(location=(port['lat'], port['lon']), radius=4, color='red', weight=2, fill=True, fill_color='red', fill_opacity=0.8, popup=HTML(f"<b>{port.get('port_name', 'Major Port')}</b>"))
    m_gdp_dist.add_layer(port_marker)

# Add legend
legend_html = HTML(
    "<div style='background: rgba(255,255,255,0.9); color: black; padding: 10px; border-radius: 8px; font-size: 12px; line-height: 1.5;'>"
    "<b>Map legend</b><br>"
    "<span style='display:block; margin-top:6px;'><span style='background:#ff7f7f; width:12px; height:12px; display:inline-block; margin-right:6px;'></span>Port marker</span>"
    "<hr style='margin:6px 0;'>"
    "<b>Country fill</b><br>"
    "<span style='display:block; margin-top:6px;'>Hue: GDP</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Blue = lower GDP</li>"
    "<li>Red = higher GDP</li>"
    "</ul>"
    "<span style='display:block; margin-top:6px;'>Brightness: distance to nearest port</span>"
    "<ul style='margin:4px 0 0 16px; padding:0;'>"
    "<li>Brighter = closer to a major port</li>"
    "<li>Darker = farther from a major port</li>"
    "</ul>"
    "</div>"
)
legend_control = WidgetControl(widget=legend_html, position='bottomright')
m_gdp_dist.add_control(legend_control)

m_gdp_dist

Map(center=[np.float64(11.858591944881889), np.float64(17.16719923622047)], controls=(ZoomControl(options=['po…

## 16. Save Updated Data

In [88]:
output_file_gdp = 'processed_food_security_analysis_with_gdp.csv'
merged_gdp_df.to_csv(output_file_gdp, index=False)
print(f"\nAnalysis with GDP ready! Results saved to '{output_file_gdp}'")


Analysis with GDP ready! Results saved to 'processed_food_security_analysis_with_gdp.csv'


In [89]:
# Scatter plot of GDP vs distance to nearest port for developing countries, colored by undernourishment
colored_chart = alt.Chart(merged_gdp_df).mark_point(filled=True, size=60).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    color=alt.Color('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'GDP_billions_USD', 'dist_to_nearest_port_km', 'latest_undernourishment']
).properties(
    width=600,
    height=400,
    title='GDP vs Distance to Nearest Port Colored by Undernourishment'
).interactive()
colored_chart.show()

alt.Chart(...)

## 17. k-NN Regression Analysis
Apply k-Nearest Neighbors regression to model the non-linear relationship between variables.

In [90]:
# Try k-NN Regression to see if we can find a non-linear relationship between distance to port and undernourishment, or GDP and undernourishment

# 1. Clean data for regression
reg_df = merged_gdp_df[['dist_to_nearest_port_km', 'GDP_billions_USD', 'latest_undernourishment']].copy()
reg_df['GDP_billions_USD'] = pd.to_numeric(reg_df['GDP_billions_USD'], errors='coerce')
reg_df = reg_df.dropna()

# 2. k-NN Regression for Distance to Port vs Undernourishment -- Used Gemini to assist in writing this function
def get_knn_predictions(X, y, k=10):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values.reshape(-1, 1))

    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_scaled, y)

    # Create smooth range for plotting
    x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
    x_range_scaled = scaler.transform(x_range)
    y_pred = knn.predict(x_range_scaled)

    return x_range.flatten(), y_pred

In [91]:
# k-NN Regression: Distance to Port vs Undernourishment
x_dist, y_dist_pred = get_knn_predictions(reg_df['dist_to_nearest_port_km'], reg_df['latest_undernourishment'], k=15)
dist_pred_df = pd.DataFrame({'dist_to_nearest_port_km': x_dist, 'predicted_undernourishment': y_dist_pred})

scatter_dist = alt.Chart(reg_df).mark_point(filled=True, size=60, opacity=0.5).encode(
    x=alt.X('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    tooltip=['dist_to_nearest_port_km', 'latest_undernourishment']
)

line_dist = alt.Chart(dist_pred_df).mark_line(color='red', size=3).encode(
    x='dist_to_nearest_port_km:Q',
    y='predicted_undernourishment:Q'
)

(scatter_dist + line_dist).properties(
    width=600,
    height=400,
    title='k-NN Regression: Distance to Port vs Undernourishment'
).interactive()

alt.LayerChart(...)

In [92]:
# k-NN Regression: GDP vs Undernourishment
x_gdp, y_gdp_pred = get_knn_predictions(reg_df['GDP_billions_USD'], reg_df['latest_undernourishment'], k=15)
gdp_pred_df = pd.DataFrame({'GDP_billions_USD': x_gdp, 'predicted_undernourishment': y_gdp_pred})

scatter_gdp = alt.Chart(reg_df).mark_point(filled=True, size=60, opacity=0.5).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    tooltip=['GDP_billions_USD', 'latest_undernourishment']
)

line_gdp = alt.Chart(gdp_pred_df).mark_line(color='red', size=3).encode(
    x='GDP_billions_USD:Q',
    y='predicted_undernourishment:Q'
)

(scatter_gdp + line_gdp).properties(
    width=600,
    height=400,
    title='k-NN Regression: GDP vs Undernourishment'
).interactive()

alt.LayerChart(...)

## 17. K-means Clustering

In [93]:
# Utilized assistance from GitHub Copilot to setup Scalar and KMeans models, and prepare clustered data for display
# Prepare features for clustering
features = ['latest_undernourishment', 'dist_to_nearest_port_km', 'GDP_billions_USD']
X = merged_gdp_df[features].dropna()

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit K-means with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to the dataframe
X_clustered = X.copy()
X_clustered['cluster'] = clusters

# Merge back to get countries
merged_gdp_df_clustered = merged_gdp_df.merge(X_clustered[['cluster']], left_index=True, right_index=True, how='left')

# Display cluster centers (inverse scaled)
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(cluster_centers, columns=features)
print("Cluster Centers:")
print(centers_df)

# Visualize clusters
chart_cluster_gdp_und = alt.Chart(merged_gdp_df_clustered.dropna(subset=['cluster'])).mark_circle(size=60).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('cluster:N', title='Cluster'),
    tooltip=['Country', 'GDP_billions_USD', 'latest_undernourishment', 'cluster']
).properties(
    width=400,
    height=300,
    title='Clusters: GDP vs Undernourishment'
)

chart_cluster_gdp_dist = alt.Chart(merged_gdp_df_clustered.dropna(subset=['cluster'])).mark_circle(size=60).encode(
    x=alt.X('GDP_billions_USD:Q', title='GDP (Billions USD)'),
    y=alt.Y('dist_to_nearest_port_km:Q', title='Distance to Nearest Port (km)'),
    color=alt.Color('cluster:N', title='Cluster'),
    tooltip=['Country', 'GDP_billions_USD', 'dist_to_nearest_port_km', 'cluster']
).properties(
    width=400,
    height=300,
    title='Clusters: GDP vs Distance to Port'
)

chart_cluster_dist_und = alt.Chart(merged_gdp_df_clustered.dropna(subset=['cluster'])).mark_circle(size=60).encode(
    x=alt.X('dist_to_nearest_port_km:Q', title='Distance to Nearest Port (km)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('cluster:N', title='Cluster'),
    tooltip=['Country', 'GDP_billions_USD', 'latest_undernourishment', 'cluster']
).properties(
    width=400,
    height=300,
    title='Clusters: Distance to Port vs Undernourishment'
)

chart_cluster_gdp_und | chart_cluster_dist_und | chart_cluster_gdp_dist

Cluster Centers:
   latest_undernourishment  dist_to_nearest_port_km  GDP_billions_USD
0                26.321739               970.786724         37.143043
1                 6.535484              1042.140236         52.527645
2                 6.257143               404.753813        473.430286


alt.HConcatChart(...)

## 18. Save Clustered Data

In [94]:
output_file_clustered = 'processed_food_security_analysis_with_gdp_clustered.csv'
merged_gdp_df_clustered.to_csv(output_file_clustered, index=False)
print(f"\nClustered analysis ready! Results saved to '{output_file_clustered}'")


Clustered analysis ready! Results saved to 'processed_food_security_analysis_with_gdp_clustered.csv'


In [96]:
# Map developed with assistance from GitHub Copilot

# 1. Define a simple color mapping for your 3 clusters
# Cluster 0: High Undernourishment (Red)
# Cluster 1: Low Undernourishment, Higher Distance (Orange/Yellow)
# Cluster 2: High GDP, Low Distance (Green)
cluster_colors = {
    0: '#e74c3c', # Red
    1: '#f1c40f', # Yellow
    2: '#2ecc71', # Green
    None: '#cccccc' # Gray for no data
}

# 2. Create the Map
map_center = (merged_gdp_df_clustered['latitude'].mean(), merged_gdp_df_clustered['longitude'].mean())
m_clusters = Map(center=map_center, zoom=2, basemap=basemaps.CartoDB.Positron)

# 3. Merge geometries with clustered data
world_clusters = world.merge(
    merged_gdp_df_clustered[['Country', 'cluster', 'latest_undernourishment', 'dist_to_nearest_port_km', 'GDP_billions_USD']],
    left_on='name', right_on='Country', how='left'
)

# 4. Add Country Polygons
for idx, row in world_clusters.iterrows():
    cluster_id = row['cluster']
    # Determine color based on cluster
    color_hex = cluster_colors.get(cluster_id if pd.notna(cluster_id) else None)

    popup_text = (
        f"<b>Country:</b> {row['name']}<br>"
        f"<b>Cluster:</b> {int(cluster_id) if pd.notna(cluster_id) else 'N/A'}<br>"
        f"<b>Undernourishment:</b> {row['latest_undernourishment']}%<br>"
        f"<b>GDP:</b> ${row['GDP_billions_USD']:.1f}B"
    )

    if row['geometry']:
        # standard handling for Polygon/MultiPolygon
        if row['geometry'].geom_type == 'Polygon':
            coords = [(lat, lon) for lon, lat in row['geometry'].exterior.coords]
            poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
            m_clusters.add_layer(poly)
        elif row['geometry'].geom_type == 'MultiPolygon':
            for geom in row['geometry'].geoms:
                coords = [(lat, lon) for lon, lat in geom.exterior.coords]
                poly = Polygon(locations=coords, color='gray', weight=1, fill_color=color_hex, fill_opacity=0.7, popup=HTML(popup_text))
                m_clusters.add_layer(poly)

# Add ports
for _, port in ports_df.iterrows():
    port_marker = CircleMarker(location=(port['lat'], port['lon']), radius=4, color='red', weight=2, fill=True, fill_color='red', fill_opacity=0.8, popup=HTML(f"<b>{port.get('port_name', 'Major Port')}</b>"))
    m_clusters.add_layer(port_marker)

# 5. Add a Legend for the Clusters
legend_html = HTML("""
<div style='background: rgba(255,255,255,0.9); color: black; padding: 10px; border-radius: 8px; font-size: 12px; line-height: 1.5; border: 1px solid gray;'>
    <b>Map Legend</b><br>
    <span style='display:block; margin-top:6px;'>
        <span style='background:#ff7f7f; width:12px; height:12px; display:inline-block; margin-right:6px; border-radius:50%;'></span>
        Major Port
    </span>
    <hr style='margin:6px 0;'>
    <b>K-Means Clusters</b><br>
    <i style="background: #e74c3c; width: 12px; height: 12px; display: inline-block; margin-right:6px;"></i> Cluster 0 (High Risk)<br>
    <i style="background: #f1c40f; width: 12px; height: 12px; display: inline-block; margin-right:6px;"></i> Cluster 1 (Mid-Tier)<br>
    <i style="background: #2ecc71; width: 12px; height: 12px; display: inline-block; margin-right:6px;"></i> Cluster 2 (Strong Economy)<br>
</div>
""")
m_clusters.add_control(WidgetControl(widget=legend_html, position='bottomright'))

m_clusters

Map(center=[np.float64(11.858591944881889), np.float64(17.16719923622047)], controls=(ZoomControl(options=['po…